# LSB method in the spatial domain (encrypted)

## Module imports

In [77]:
import textwrap
import numpy as np
from PIL import Image
from Crypto.Hash import HMAC, SHA256
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from math import log10, sqrt

## Functions for secret message file processing

In [78]:
# convert byte array to bit string
def bytes_to_bits(bytes):
            
    bits = ''.join(f"{byte:08b}" for byte in bytes)

    return bits


# convert bit string to byte array
def bits_to_bytes(bits):

    val = int(bits, 2)
    n_B = len(bits) // 8

    return val.to_bytes(n_B, byteorder="big")


# zero pad and wrap to equal k length blocks
def pad_wrap(bits, k):

    mod = len(bits) % k
    if mod != 0:
        bits += "0" * (k-mod)
    
    return textwrap.wrap(bits, k)


# merge binary blocks to string and unpad 0s
def merge_unpad(bits):

    bits = "".join(bits)
    mod = len(bits) % 8

    return bits[:len(bits) - mod]


def encrypt(plaintext, key):

    iv = get_random_bytes(16)
    cipher = AES.new(key, AES.MODE_CFB, iv)
    ciphertext = cipher.encrypt(plaintext)

    return iv, ciphertext


def decrypt(iv, ciphertext, key):

    cipher = AES.new(key, AES.MODE_CFB, iv)
    plaintext = cipher.decrypt(ciphertext)
    
    return plaintext

## Image processing functions

In [79]:
def PSNR(original, noisy):

    mse = np.mean((original-noisy) ** 2)

    # set rounded value for PSNR
    # MSE = 0 --> PSNR goes to infinity
    if(mse == 0):

        return 100, 0
    
    max_pixel = 255.0
    psnr = 20 * log10(max_pixel / sqrt(mse))
    
    return psnr, mse


# 1st method
# LSB substitution method
def substitute_lsb(pv, b):

    pv &= ~ np.uint8(1)
    pv |=   np.uint8(b)

    return pv


# 2nd method
# LSB +/-1 embedding
def match_lsb(pv):

    match pv:
        case 0:
            pv += np.uint8(1)
        case 255:
            pv -= np.uint8(1)
        case _:
            pv += np.random.choice([-1, 1])

    return pv

## Main ```embed_pixels``` and ```extract_pixels``` functions

In [80]:
def embed_pixels(array, ids, bits):

    for b, i in zip(bits, ids):

        pv = array[i]
        if pv%2 != int(b):

            #pv1 = substitute_lsb(pv, b)    
            pv1 = match_lsb(pv)
            array[i] = pv1

    return array


def extract_pixels(array, ids):

    bits = []
    for i in ids:

        pv = array[i]
        b = pv % 2
        bits.append(str(b))
        
    return bits

## Embedding process

### Part 1: secret message file processing

In [81]:
f_name = b"message.txt"

with open(f_name, "rb") as f:
    f_data = f.read()

# data: file name + contents
data = len(f_name).to_bytes(1) + f_name + f_data
# data size stored in 3B
size = len(data).to_bytes(3)

# for encryption
# first 3B of ciphertext will be encrypted size
plaintext = size + data

print(plaintext)

aes_passw = b"password123"
h = SHA256.new(data= aes_passw)
key = h.digest()

mac = HMAC.new(key, plaintext, SHA256).digest()
iv, ciphertext = encrypt(plaintext, key)

# part 1: iv + first 3B of ciphertext (size)
p1_B = iv + ciphertext[:3]
p2_B = ciphertext[3:] + mac

print("mac: ", mac)
print("p1_B: ", p1_B)
print("p2_B: ", p2_B)

p1_b = bytes_to_bits(p1_B)
p1_b = pad_wrap(p1_b,1)

p2_b = bytes_to_bits(p2_B)
p2_b = pad_wrap(p2_b,1)

n_pts1 = len(p1_b)
n_pts2 = len(p2_b)

b'\x00\x00\x84\x0bmessage.txt"Listen very carefully, I shall say this only once!"\n\n                                   - Michele Dubois, \'Allo \'Allo!\n'
mac:  b'\x05\x11\r[\x9b\x9f\x86\xce\xee}\x84\xaez\x08\xe5\x90.\x05\xf5)\x0f-\x11g\xb0\x01(\x95\xf71if'
p1_B:  b'\xcfy\x87\xe1\xf4\xde\xb2JF\x9a\xb5}X\x93\x02\x1f3\x8a\xb7'
p2_B:  b'\x95u\xe3i\xb9\xb2\xc2\xd6\xdd?S{\xb3\xcf\xf4\xbc\xf7t\xf1\x15\x8fVtM\xf9\xae\xea\x03v>\x1781x\xd8\xcbz\x822\xb5\xfcj\xf6\xacB\x912\xec\x1f\xb5N\x0e\xfc=\x87\x8c\x86\x8d\xc9\x1bQ\x7fe\xd5,a\xfa\xa1x@\xcc/\xf3j.er\xb6%B\xaf\xea\xa0\x9f\x1c\xec\t\xaa\x82\x93j\xa8\x8bW\xectm\t\x89N\xc1\xf2\xa8\xc0\xc4~\xc3\xf5\xed\xc4l\xe4\xf6\xa5\x91\xb6\xc9\xb1\xc7e]\xeeX\xac9y\x0b\x10=\xb6\x9db\x05\x11\r[\x9b\x9f\x86\xce\xee}\x84\xaez\x08\xe5\x90.\x05\xf5)\x0f-\x11g\xb0\x01(\x95\xf71if'


### Part 2: image processing 

In [82]:
cover = Image.open("lenna.bmp")
c_arr = np.array(cover)

# flat cover pixels array
c_arr1 = c_arr.flatten()

rng_passw = b"stegopass"
h = SHA256.new(data= rng_passw)

seed = int.from_bytes(h.digest())
rng = np.random.default_rng(seed)

# n of image pixel points
n_pts = len(c_arr1)
# initial indices array
ids_arr0 = np.arange(n_pts)

# indices selection for embedding p1_b
sel1 = rng.choice(ids_arr0, n_pts1, 0)

# indices selection for embedding p2_b
mask = ~np.isin(ids_arr0, sel1)
ids_arr1 = ids_arr0[mask]
sel2 = rng.choice(ids_arr1, n_pts2, 0)

# flat stego pixels array after embedding
s_arr1 = embed_pixels(c_arr1, sel1, p1_b)
s_arr1 = embed_pixels(c_arr1, sel2, p2_b)

shape = c_arr.shape
s_arr = s_arr1.reshape(shape)

stego = Image.fromarray(s_arr)
stego.save("stego2.png")

## Extraction example

### Part 1: extracting `p1_B` from stego image

In [83]:
stego = Image.open("stego2.png")
s_arr = np.array(stego)
s_arr1 = s_arr.flatten()

rng_passw = b"stegopass"
h = SHA256.new(data= rng_passw)

seed = int.from_bytes(h.digest())
rng = np.random.default_rng(seed)

# n of image pixel points
n_pts = len(s_arr1)
# initial indices array
ids_arr0 = np.arange(n_pts)

# part 1: iv (16B) + size (3B)
#########################################
# n of embedded points for p1_b
n_pts1 = 19 * 8

# indices selection with embedded p1_b
sel1 = rng.choice(ids_arr0, n_pts1, 0)

p1_b = extract_pixels(s_arr1, sel1)
p1_B = bits_to_bytes(merge_unpad(p1_b))

# separate iv for decryption
iv = p1_B[:16]

aes_passw = b"password123"
h = SHA256.new(data= aes_passw)
key = h.digest()

# ciphertext: 3B of encrypted size
ciphertext = p1_B[16:]
size = decrypt(iv, ciphertext, key)
size = int.from_bytes(size)

print(size)

132


### Part 2: extracting `p2_B` from stego image

It is important to run the following cell only once after the previous part! Otherwise, the pseudo random number generator will generate different positions each time (resulting in extraction of random bytes for p2_B). If this happens, run the Part 1 again to initialize the RNG and start over (or simply merge part 1 and part 2 in a single cell) Extraction of wrong bytes will result in a failed HMAC authentication.

In [84]:
# n of embedded points for p2_b
# extracted size + 32B (MAC)
n_pts2 = (size+32) * 8

# indices selection with embedded p2_b
mask = ~np.isin(ids_arr0, sel1)
ids_arr1 = ids_arr0[mask]
sel2 = rng.choice(ids_arr1, n_pts2, 0)

p2_b = extract_pixels(s_arr1, sel2)
p2_B = bits_to_bytes(merge_unpad(p2_b))

# update ciphertext with extracted p2_B
ciphertext += p2_B[:-32]
# separate 32 mac bytes
mac = p2_B[-32:]

print("\nciphertext: ", ciphertext)

plaintext = decrypt(iv, ciphertext, key)
HMAC.new(key, plaintext, SHA256).verify(mac)

data = plaintext[3:]


ciphertext:  b'3\x8a\xb7\x95u\xe3i\xb9\xb2\xc2\xd6\xdd?S{\xb3\xcf\xf4\xbc\xf7t\xf1\x15\x8fVtM\xf9\xae\xea\x03v>\x1781x\xd8\xcbz\x822\xb5\xfcj\xf6\xacB\x912\xec\x1f\xb5N\x0e\xfc=\x87\x8c\x86\x8d\xc9\x1bQ\x7fe\xd5,a\xfa\xa1x@\xcc/\xf3j.er\xb6%B\xaf\xea\xa0\x9f\x1c\xec\t\xaa\x82\x93j\xa8\x8bW\xectm\t\x89N\xc1\xf2\xa8\xc0\xc4~\xc3\xf5\xed\xc4l\xe4\xf6\xa5\x91\xb6\xc9\xb1\xc7e]\xeeX\xac9y\x0b\x10=\xb6\x9db'


In [85]:
name_l = data[0]
f_name = data[1:1+name_l]
f_data = data[1+name_l: ]
print(f"extracted file names is: {f_name}")
print(f"extracted file content is: \n{f_data}")

extracted file names is: b'message.txt'
extracted file content is: 
b'"Listen very carefully, I shall say this only once!"\n\n                                   - Michele Dubois, \'Allo \'Allo!\n'


## Signal analysis

In [86]:
cover = Image.open("lenna.bmp")
c_arr = np.asarray(cover)

stego = Image.open("stego2.png")
s_arr = np.asarray(stego)

psnr, mse = PSNR(c_arr, s_arr)

print(f"PSNR value {psnr}")
print(f"MSE value: {mse}")

PSNR value 78.47804916839141
MSE value: 0.00092315673828125


## Cover image embedding capacity

In [87]:
cover = Image.open("lenna.bmp")
c_arr = np.asarray(cover)
c_arr1 = c_arr.flatten()

n_pts = len(c_arr1)
# kapacitet u bajtovima
byte_cap = n_pts // 8

print(f"cover capacity: {byte_cap} B")

cover capacity: 98304 B
